In [1]:
# Processed E-commerce Dataset (Day 9)
import pandas as pd

# Load source datasets
customers = pd.read_csv("Day9_Customers.csv")
orders = pd.read_csv("Day9_Orders.csv")
products = pd.read_csv("Day9_Products.csv")

# Demonstrate concat() by stacking sample rows from the same structure
customer_sample = customers[["Customer_ID", "Customer_Name"]].head(3)
customer_tail = customers[["Customer_ID", "Customer_Name"]].tail(3)
concat_demo = pd.concat([customer_sample, customer_tail], ignore_index=True)

# Merge orders with customers and products
orders_with_customers = orders.merge(customers, on="Customer_ID", how="left")
ecommerce_df = orders_with_customers.merge(products, on="Product_ID", how="left")

# Convert date and extract useful date fields
# DateTime operations
for col in ["Order_Date"]:
    ecommerce_df[col] = pd.to_datetime(ecommerce_df[col])

ecommerce_df["Order_Year"] = ecommerce_df["Order_Date"].dt.year
ecommerce_df["Order_Month"] = ecommerce_df["Order_Date"].dt.month_name()
ecommerce_df["Order_Weekday"] = ecommerce_df["Order_Date"].dt.day_name()

# Create useful derived columns using apply()
def customer_segment(membership_type):
    if membership_type == "Premium":
        return "Premium"
    elif membership_type == "Regular":
        return "Regular"
    return "New"

ecommerce_df["Customer_Segment"] = ecommerce_df["Membership_Type"].apply(customer_segment)

ecommerce_df["Order_Value"] = (ecommerce_df["Quantity"] * ecommerce_df["Unit_Price"]).round(2)

def sales_bucket(order_value):
    if order_value >= 10000:
        return "High"
    elif order_value >= 5000:
        return "Medium"
    return "Low"

ecommerce_df["Sales_Bucket"] = ecommerce_df["Order_Value"].apply(sales_bucket)

# Reorder columns into a clean final dataset
final_columns = [
    "Order_ID", "Order_Date", "Order_Year", "Order_Month", "Order_Weekday",
    "Customer_ID", "Customer_Name", "City", "Region", "Membership_Type", "Customer_Segment",
    "Product_ID", "Product_Name", "Category", "Brand", "Unit_Price", "Quantity", "Order_Value",
    "Payment_Method", "Order_Status", "Sales_Bucket"
]

processed_df = ecommerce_df[final_columns].sort_values("Order_Date").reset_index(drop=True)

# Save processed dataset to CSV
processed_df.to_csv("processed_ecommerce_dataset.csv", index=False)

print("Concat demonstration sample:")
print(concat_demo.to_string(index=False))
print("\nProcessed dataset preview:")
print(processed_df.head(10).to_string(index=False))
print(f"\nFinal dataset rows: {len(processed_df)}")
print("CSV saved as processed_ecommerce_dataset.csv")


Concat demonstration sample:
Customer_ID  Customer_Name
       C001   Aarav Sharma
       C002      Zoya Khan
       C003    Rohan Mehta
       C028     Inaya Khan
       C029 Sameer Qureshi
       C030    Riya Kapoor

Processed dataset preview:
Order_ID Order_Date  Order_Year Order_Month Order_Weekday Customer_ID Customer_Name       City Region Membership_Type Customer_Segment Product_ID            Product_Name       Category     Brand  Unit_Price  Quantity  Order_Value   Payment_Method Order_Status Sales_Bucket
   O0051 2026-01-01        2026     January      Thursday        C008    Sara Ahmed  Hyderabad  South         Premium          Premium       P017                Yoga Mat         Sports   FitLife         899         4         3596      Net Banking    Delivered          Low
   O0091 2026-01-01        2026     January      Thursday        C002     Zoya Khan      Delhi  North         Regular          Regular       P011               Air Fryer Home & Kitchen CookSmart        4999  